In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import contextily as ctx
import numpy as np
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px
import pyproj
from plotly.subplots import make_subplots
pd.options.mode.chained_assignment = None
#FR: © EuroGeographics pour les limites administratives
#https://ec.europa.eu/eurostat/web/gisco/geodata/administrative-units/communes

In [ ]:
senateurs = pd.read_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/senateur_for_map.csv")


In [ ]:
# Reste dans la ville sauf mention contraire 
# prendre en compte les morts
#prendre lieu précédent par défaut
# agréger les différentes participations assemblées révolution
# grand conseil d'administration : vérifier si récupération de plusieurs dates 
# nombre de décorations 

In [ ]:
# activité famille 
# interprétation des cartes

In [ ]:
from geopy.geocoders import Nominatim
geolocator = Nominatim(user_agent="test_senateur")

In [ ]:
result = geolocator.geocode("Vaucluse", language='fr')

In [ ]:
from geopy.extra.rate_limiter import RateLimiter
from geopy.distance import geodesic
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

In [ ]:
years = list(range(1789, 1816))

In [ ]:
senateurs.columns

In [ ]:
geocode_paris = geocode('Paris')

In [ ]:
dic_lieux = {}

In [ ]:
def geocode_to_apply(x):
    if x in dic_lieux.keys():
        return dic_lieux[x]
    else:
        result = geocode(x)
        dic_lieux[x]=result
        return result

In [ ]:
latitude_paris = geocode_paris.latitude
longitude_paris = geocode_paris.longitude
def compute_distance(x):
    if not x:
        return None
    return (geodesic((latitude_paris, longitude_paris), (x.latitude, x.longitude)).kilometers)

In [ ]:
def compute_distance_to_birth(personne, year):
    if (not personne['latitude'+str(annee)]) or pd.isna(personne['latitude'+str(annee)]) or pd.isna(personne['latitude_naiss']):
        return None
    return (geodesic((personne['latitude_naiss'], personne['longitude_naiss']), 
                     (personne['latitude'+str(annee)], personne['longitude'+str(annee)])).kilometers)

In [ ]:
dic_annee={}
pd.options.mode.chained_assignment = None
senateurs["result_naiss"] = senateurs["ville naissance"].apply(geocode_to_apply)
senateurs['latitude_naiss'] = senateurs['result_naiss'].apply(lambda x : x.latitude if x else None)
senateurs['longitude_naiss'] = senateurs['result_naiss'].apply(lambda x : x.longitude if x else None)
for annee in years:#senateurs[senateurs[str(annee)].notna() & 
    senateurs["nom_local"+str(annee)].fillna('', inplace = True)
    senateurs['result'+str(annee)] = senateurs["nom_local"+str(annee)].apply(geocode_to_apply)
    senateurs['distance'+str(annee)] = senateurs['result'+str(annee)].apply(compute_distance)
    senateurs['latitude'+str(annee)] = senateurs['result'+str(annee)].apply(lambda x : x.latitude if x else None)
    senateurs['longitude'+str(annee)] = senateurs['result'+str(annee)].apply(lambda x : x.longitude if x else None)
    senateurs['distance_to_naiss'+str(annee)] = senateurs.apply(compute_distance_to_birth, axis = 1, args = [annee])
    

In [ ]:
senateurs.to_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/senateur_with_lat.csv")

In [ ]:
senateurs = pd.read_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/senateur_with_lat.csv")

In [ ]:
dic_annee={}
for annee in years:#senateurs[senateurs[str(annee)].notna() &
    list_name = ["nom_local"+str(annee), "nom", "annee nomin", #"position_sociale_restreint",
                 'latitude'+str(annee), 'longitude'+str(annee), 'distance_to_naiss'+str(annee)]
    senateur_year = senateurs[senateurs["nom_local"+str(annee)]!="Paris"][list_name]
    senateur_year["en exercice"] = (senateur_year["annee nomin"]<=annee).astype(int)
    dic_annee[annee]=senateur_year

In [ ]:

for annee in years:#senateurs[senateurs[str(annee)].notna() & 
    dic_annee[annee]["nom_local"] = dic_annee[annee]["nom_local" + str(annee)]
    dic_annee[annee]["nombre_senateurs"] = dic_annee[annee].groupby("nom_local").nom.transform("nunique")


In [ ]:
dic_annee[1800].columns

In [ ]:
year=1795

In [ ]:
minx, miny, maxx, maxy = -10.4477, 34.5949, 34.3694, 71.0313

In [ ]:
def update_layout_default(fig, year = 1800):
        return fig.update_layout(
        height = 500,
        width = 500,
        title = go.layout.Title(
            text = 'Senateurs en {}'.format(year)),
        geo = go.layout.Geo(
            resolution = 110,
            scope = 'europe',
            showframe = False,
            showcoastlines = True,
            landcolor = "rgb(229, 229, 229)",
            #countrycolor = None,
            showcountries = False,
            coastlinecolor = "white",
            projection_type = 'natural earth',#'mercator
            lonaxis_range= [ minx, maxx -5],
            lataxis_range= [ miny, maxy ],
        ),
        geo2 = go.layout.Geo(
            #scope = 'europe',
            showframe = False,
            landcolor = "rgb(229, 229, 229)",
            showcountries = False,
            #domain = dict(x = [ 0, 0.6 ], y = [ 0, 0.6 ]),
            bgcolor = 'rgba(255, 255, 255, 0.0)',
        ),
        legend_traceorder = 'reversed'
    )
    

In [ ]:
colors = ['red', 'blue', 'green', 'brown', 'purple', 'coral', 'orange',
            'darkkhaki', 'darkmagenta', 'darkolivegreen', 'darkorange',
            'darkorchid', 'darkred', 'darksalmon', 'darkseagreen',
            'darkslateblue', 'darkslategray',
            'darkturquoise', 'darkviolet', 'deeppink', 'deepskyblue', 'dimgrey', 'dodgerblue', 'firebrick',
            'floralwhite', 'forestgreen', 'fuchsia', 'gainsboro', 'gold', 'goldenrod', 'gray', 'grey']
markers = ['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'pentagon', 'hexagram',
           'star', 'hourglass', 'bowtie', 'asterisk', 'hash', 'triangle-down', 'octagon', 'cross-open', 'triangle-nw-dot',
          'pentagon-dot', 'hexagon-dot']

In [ ]:
### Préparer cas où personnes des deux types au même endroit

In [ ]:

def plot_from_year(year = 1800, column_to_separe='en exercice', column_to_separe2=None, save = False):#, position = [1, 1], big_fig = make_subplots(
        #rows=1, cols=1)):
    #row= position[0]
    #col = position[1]
    df = dic_annee[year]

    fig = go.Figure()
    i = 0
    for value in df[column_to_separe].unique():
        if (value!= value):
            value = 'Inconnu'
        df_part = df[df[column_to_separe]==value]
        if column_to_separe2 is None:
            df_part["nombre_senateurs"] = df_part.groupby("nom_local").nom.transform("nunique").fillna(0)
            fig.add_trace(
                    go.Scattergeo(
                    lon = df_part['longitude'+str(year)],
                    lat = df_part['latitude'+str(year)],
                    text = df_part['nom_local'],
                    showlegend = True,
                    name = str(value),
                    opacity = 0.7,
                    marker = dict(
                        size = df_part['nombre_senateurs']*5,
                        line_width = 0,
                        color = colors[i],
                    )))#, row=row, col=col)
        else:
            j=0
            for value2 in df[column_to_separe2].unique():
                if (value2!= value2):
                    value2 = 'Inconnu'
                df_part2 = df_part[df_part[column_to_separe2]==value2]
                df_part2["nombre_senateurs"] = df_part2.groupby("nom_local").nom.transform("nunique")
                df_part2["nombre_senateurs"]= df_part2["nombre_senateurs"].fillna(0)
                fig.add_trace(go.Scattergeo(
                        lon = df_part2['longitude'+str(year)],
                        lat = df_part2['latitude'+str(year)],
                        text = df_part2['nom_local'],
                        opacity = 0.7,
                        showlegend = True,
                        name = str(value2),
                        marker = dict(
                            size = df_part2['nombre_senateurs'],
                            line_width = 0,
                            symbol=markers[j],
                            color = colors[i],
                        ))#, row=row, col=col)
                             )
                j+=1
        i+=1
    update_layout_default(fig, year = year)
    if save:
        fig.write_image(save)

    return fig

In [ ]:
# Eylau qui reste, Egypte aussi

In [ ]:
for year in years:
    plot_from_year(year = year, save = 'C:/Users/sylva/OneDrive/Bureau/senat/Data/plot_'+ str(year)+'.png')


In [ ]:
plot_from_year(year = 1789, save = 'C:/Users/sylva/OneDrive/Bureau/senat/Data/plot_1789.png')


In [ ]:
plot_from_year(year = 1800)


In [ ]:
plot_from_year(year = 1789, save = 'https://onedrive.live.com/?authkey=%21AB5dud8S3DBRfXE&id=4C6BFAF1D3A2778E%21211&cid=4C6BFAF1D3A2778E/plot_1789.png')


In [ ]:
! pip install -U kaleido

In [ ]:
plot_from_year(year = 1806, column_to_separe = 'position_sociale')

In [ ]:

df = dic_annee[1800]
df.to_crs(pyproj.CRS.from_epsg(4326), inplace=True)
fig = px.scatter_geo(df, geojson='geometry', color="en exercice",
                      size="nombre_senateurs")
fig.show()


In [ ]:


# Initialize figure with subplots
def plot_several_year(years):
    cols, rows = 2,int(len(years)/2+0.5)
    fig = make_subplots(
        rows=rows, cols=cols, subplot_titles=(years)
    )

    # Add traces
    row = 1
    col = 1
    for annee in years:
        row = (row - 1)*2
        if row == 1:
            col+=1
          
        plot_from_year(big_fig = fig, year = annee, column_to_separe='en exercice', column_to_separe2=None, position = [row, col])
    
    # Update xaxis properties
    # fig.update_xaxes(title_text="xaxis 1 title", row=1, col=1)
    # fig.update_xaxes(title_text="xaxis 2 title", range=[10, 50], row=1, col=2)
    # fig.update_xaxes(title_text="xaxis 3 title", showgrid=False, row=2, col=1)
    # fig.update_xaxes(title_text="xaxis 4 title", type="log", row=2, col=2)
    
    # # Update yaxis properties
    # fig.update_yaxes(title_text="yaxis 1 title", row=1, col=1)
    # fig.update_yaxes(title_text="yaxis 2 title", range=[40, 80], row=1, col=2)
    # fig.update_yaxes(title_text="yaxis 3 title", showgrid=False, row=2, col=1)
    # fig.update_yaxes(title_text="yaxis 4 title", row=2, col=2)
    
    # # Update title and height
    # fig.update_layout(title_text="Customizing Subplot Axes", height=700)
    
    fig.show()

In [ ]:
plot_several_year([1789, 1804])

In [ ]:
years_consulat = list(range(1799, 1805))
def plot_several_years(years):
    cols, rows = 2,int(len(years)/2+0.5)  #num of subplots <= (cols x rows)
    # create figure with array of axes
    fig, axs = plt.subplots(nrows=rows, ncols=cols)
    fig.set_size_inches(6, 10)  #set it big enough for all subplots
    
    count = 0
    for irow in range(axs.shape[0]):
        for icol in range(axs.shape[1]):
            #print(icol, irow)
            if count<len(years):
                # plot that country on current axes
                gdf_country.plot(figsize=(20,15), color="gray", ax=axs[irow][icol])
                #dic_annee[years[count]][dic_annee[years[count]]["en exercice"]].plot(figsize=(20,15), ax=axs[irow][icol], color="yellow", aspect = 1, markersize = dic_annee[years[count]]["nombre_senateurs"]*20)
                #dic_annee[years[count]][~dic_annee[years[count]]["en exercice"]].plot(figsize=(20,15), ax=axs[irow][icol], color="green", aspect = 1, markersize = dic_annee[years[count]]["nombre_senateurs"]*20)
                dic_annee[years[count]].plot(column="en exercice", figsize=(20,15), ax=axs[irow][icol], aspect = 1, markersize = dic_annee[years[count]]["nombre_senateurs"]*20)
                #, column="en exercice"
                #senateurs["geometry"] = senateurs["coord"+str(years[count])]#[["coord"+str(years[count]), "nom_local"+str(years[count])]]
                #senateurs.plot(figsize=(20,15), ax=axs[irow][icol], color="yellow", aspect = 1)
                axs[irow][icol].set_xlim(minx, maxx)
                axs[irow][icol].set_ylim(miny, maxy)
                #world[ world['iso_a3'] == years[count] ].plot(ax=axs[irow][icol])
                axs[irow][icol].set_title('année : '+str(years[count]))
                count +=1
            else:
                # hide extra axes
                axs[irow][icol].set_visible(False)
    #plt.savefig("C:/Users/sylva/OneDrive/Bureau/senat/plot/map_year")
    plt.show()

In [ ]:
plot_several_years(years_consulat)

In [ ]:
years_empire = list(range(1805, 1816))
plot_several_years(years_empire)

In [ ]:
dic_annee[years[1]]

In [ ]:
years_emp = list(range(1805, 1806))
plot_several_years(years_emp)

In [ ]:
#for annee in years:
#    senateurs_en_exercice = senateurs

In [ ]:
years_revolution = list(range(1790, 1788))
plot_several_years(years_revolution)

In [ ]:
senateurs.columns

In [ ]:
dic_annee[1799].columns

In [ ]:
liste_futur


In [ ]:
"""distance_moyenne_exercice = {}
part_voyage_exercice = {}
distance_moyenne_futur = {}
part_voyage_futur = {}
for annee in years:
    senateurs_en_exercice = senateurs[senateurs["annee nomin"]<=annee]
    futurs_senateurs = senateurs[senateurs["annee nomin"]>annee]
    liste_exercice = pd.merge(senateurs_en_exercice, dic_annee[annee], on = "nom", how = 'left')["distance"]
    if len(senateurs_en_exercice)>0:
        part_voyage_exercice[annee] = sum(liste_exercice.notna())/len(senateurs_en_exercice)
        distance_moyenne_exercice[annee] = np.mean(liste_exercice)
    liste_futur = pd.merge(futurs_senateurs, dic_annee[annee], on = "nom", how = 'left')["distance"]
    if len(futurs_senateurs)>0:
        part_voyage_futur[annee] = sum(liste_futur.notna())/len(futurs_senateurs)
        distance_moyenne_futur[annee] = np.mean(liste_futur)"""

In [ ]:
senateurs['distance_to_naiss1789']

In [ ]:
senateurs.columns

In [ ]:
pd.merge(senateurs_en_exercice, dic_annee[annee], on = "nom", how = 'left')

In [ ]:
dic_annee[annee]

In [ ]:
senateurs_en_exercice['distance_to_naiss1789']

In [ ]:
senateurs.columns

In [ ]:
distance_moyenne = pd.DataFrame(columns = ["year", "en exercice", "distance moyenne"])
distance_moyenne_naiss = pd.DataFrame(columns = ["year", "en exercice", "distance moyenne naissance"])
part_voyage = pd.DataFrame(columns = ["year", "en exercice", "part en déplacement"])
for annee in years:
    senateurs_en_exercice = senateurs[(senateurs["annee nomin"]<=annee) & (senateurs["annee deces"]>annee)]
    futurs_senateurs = senateurs[(senateurs["annee nomin"]>annee) & (senateurs["annee deces"]>annee)]
    liste_exercice = pd.merge(senateurs_en_exercice, dic_annee[annee], on = "nom", how = 'left')["distance"+str(annee)]
    liste_exercice_naiss = senateurs_en_exercice['distance_to_naiss'+str(annee)]
    if len(senateurs_en_exercice)>0:
        value = sum(liste_exercice.notna())/len(senateurs_en_exercice)
        dic_line = {"year" : annee, "en exercice": "en exercice", "part en déplacement": value}
        part_voyage = pd.concat([part_voyage, pd.DataFrame({k:[v] for k,v in dic_line.items()})])
        dic_line = {"year" : annee, "en exercice": "en exercice", "distance moyenne": np.mean(liste_exercice)}
        distance_moyenne = pd.concat([distance_moyenne, pd.DataFrame({k:[v] for k,v in dic_line.items()})])
        dic_line_naiss = {"year" : annee, "en exercice": "en exercice", "distance moyenne naiss": np.mean(liste_exercice_naiss)}
        distance_moyenne_naiss = pd.concat([distance_moyenne_naiss, pd.DataFrame({k:[v] for k,v in dic_line_naiss.items()})]) 
    liste_futur = pd.merge(futurs_senateurs, dic_annee[annee], on = "nom", how = 'left')["distance"+str(annee)]
    liste_futur_naiss = futurs_senateurs["distance_to_naiss"+str(annee)]
    if len(futurs_senateurs)>0:
        value = sum(liste_futur.notna())/len(futurs_senateurs)
        dic_line = {"year" : annee, "en exercice": "pas en exercice", "part en déplacement": value}
        part_voyage = pd.concat([part_voyage, pd.DataFrame({k:[v] for k,v in dic_line.items()})])
        dic_line = {"year" : annee, "en exercice": "pas en exercice", "distance moyenne": np.mean(liste_futur)}
        distance_moyenne = pd.concat([distance_moyenne, pd.DataFrame({k:[v] for k,v in dic_line.items()})]) 
        dic_line_naiss = {"year" : annee, "en exercice": "pas en exercice", "distance moyenne naiss": np.mean(liste_futur_naiss)}
        distance_moyenne_naiss = pd.concat([distance_moyenne_naiss, pd.DataFrame({k:[v] for k,v in dic_line_naiss.items()})]) 

In [ ]:
import plotly.express as px

"""px.line(x = part_voyage_futur.keys(), y = part_voyage_futur.values(), title = "Part des futurs sénateurs avec un voyage recensé")
px.line(x = part_voyage_exercice.keys(), y = part_voyage_exercice.values(), title = "Part des sénateurs en cours avec un voyage recensé")"""

In [ ]:
px.line(part_voyage, x= "year", y = "part en déplacement", color = 'en exercice')

In [ ]:
px.line(distance_moyenne, x= "year", y = "distance moyenne", color = 'en exercice')

In [ ]:
px.line(distance_moyenne_naiss, x= "year", y = "distance moyenne naiss", color = 'en exercice')

In [ ]:
# Regarder les âges moyens à la nomination, la répartition des types... des nouveaux sénateurs par an

In [ ]:
# régresser la distance sur le fait d'être nommé avec effet fixe individu ?

In [ ]:
# proche d'un endroit où la personne était déjà allé auparavant ? 
#-> récupérer l'endroit précédent le plus proche, rapport de distance >90%

In [ ]:
age_moyen_nouveau = {}
for annee in years:
    age_moyen_nouveau[annee] = annee - np.mean(senateurs[senateurs["annee nomin"]==annee]["annee naiss"])

In [ ]:
px.line(x = age_moyen_nouveau.keys(), y = age_moyen_nouveau.values())

In [ ]:
import bs4
import pandas as pd
from urllib import request
import re
import geopandas as gpd
import networkx as nx
import numpy as np
from mpl_toolkits.basemap import Basemap as Basemap
from pycountry_convert import country_alpha2_to_continent_code, country_name_to_country_alpha2
import geopy
from geopy.geocoders import Nominatim
from matplotlib import pyplot as plt
from ast import literal_eval
import plotly.graph_objs as go
import plotly.express as px
import math



def graph_era(b,e):
    '''
    Tracer le graphique des proportions de mathématiciens par pays entre deux dates
    '''
    #à l'aide de is_alive, on récupère les pays des mathématiciens vivants entre deux dates
    active = df_math_doc.loc[df_math_doc.index.to_series().apply(lambda name : is_alive(b,e,name))==True,
                    ['Country of citizenship']] 
    
    #on crée un dictionnaire qui recense les pays et le nombre de mathématicien qui y a travaillé sur la période
    d_cntry = {}
    for countries in active['Country of citizenship'].values:
        for country in countries:
            count = d_cntry.get(country,0) #on obtient le nombre de mathématiciens pour ce pays ou on crée une nouvelle entrée
            d_cntry[country] = count + 1 #on ajoute le nouveau mathématicien
    df_countries = pd.DataFrame(d_cntry.values(),index=d_cntry.keys(),columns=['Number of mathematicians'])
    df_countries = df_countries.reset_index().rename(columns={'index':'Country'})
    
    #on crée freq pour obtenir des fréquences au lieu des nombres, pour pouvoir faire des comparaisons
    #df_countries['Freq of mathematicians'] = df_countries['Number of mathematicians']/df_countries['Number of mathematicians'].sum()
    df_countries['log # of mathematicians'] = df_countries['Number of mathematicians'].apply(lambda x : math.log(1+x))
    #on veut à présent construire une carte
    #pour cela, on crée un geodataframe
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    world=world[['name','geometry']] #récupération des seules colonnes intéressantes
    world = world.rename(columns={'name':'Country'})
    
    df_countries = world.merge(df_countries,on='Country')
    
    #on trace la carte
    fig = px.choropleth(df_countries,
             geojson = df_countries.geometry,
             locations="Country", 
             locationmode = 'country names',
             color= 'log # of mathematicians',
            #color = 'Freq of mathematicians',
             color_continuous_scale =px.colors.sequential.Sunsetdark,
             #range_color=[0,1],
             hover_name = "Country",
             title='<br>Countries of work of mathematicians from %s to %s'%(b,e))
    return fig,df_countries
    